# Breaking the Token Ceiling, Hands-On

Companion notebook for the Sep 29, 2026 post on Marathe et al., *"Breaking the Token Ceiling: Distilling Smaller, Stronger Byte Models"* (arXiv 2609.12303).

Three things you can run on a laptop, no GPU, no downloads:

1. **The fertility tax** — how many BPE tokens the same sentence costs in four scripts (measured values, tiktoken `cl100k_base`).
2. **Entropy patching** — the BLT idea: group bytes into patches where next-byte entropy is low, split where it spikes.
3. **Marginalize-It vs End-of-Token** — a toy arithmetic demo of why exact logit conversion beats approximate marginalization.
4. **The crossover** — schematic scaling curves reproducing the paper's qualitative finding.

*Honesty labels: (1) uses real measured tokenizer output; (2) is a from-scratch reimplementation of the published mechanism on toy text; (3) is a toy example illustrating the paper's mechanism, not the paper's numbers; (4) is schematic — the +4% asymptote gap is the paper's number, crossover placement is illustrative.*

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Measured with tiktoken cl100k_base (BPE). Same meaning, four scripts.
langs  = ['English', 'Arabic', 'Chinese', 'Hindi']
sents  = ['The tokenizer is the most overlooked part of every language model.',
          'المُرمِّز هو الجزء الأكثر تجاهلاً في كل نموذج لغوي.',
          '分词器是每个语言模型中最被忽视的部分。',
          'टोकनाइज़र हर भाषा मॉडल का सबसे अनदेखा हिस्सा है।']
tokens = [12, 40, 22, 51]
nbytes = [len(s.encode('utf-8')) for s in sents]

print(f'{"lang":<9}{"tokens":>7}{"bytes":>7}{"tok/sent":>10}')
for l, t, b in zip(langs, tokens, nbytes):
    print(f'{l:<9}{t:>7}{b:>7}{t/tokens[0]:>9.2f}x')
print(f'\nHindi costs {tokens[3]/tokens[0]:.2f}x the tokens of English for the same sentence.')
print(f'Byte ratio Hindi/English: {nbytes[3]/nbytes[0]:.2f}x — bytes reflect information, not tokenizer favoritism.')

## 2. Entropy patching (the BLT mechanism)

The Byte Latent Transformer (Meta FAIR, Dec 2024 — the paper a viral Sept 2026 thread misattributed as new) groups raw bytes into **patches** using next-byte entropy: where the next byte is predictable, bytes merge into one patch; where entropy spikes (a word boundary, a surprise), a new patch starts. Below: a toy 27-symbol byte-bigram (lowercase a-z + space) fit on an English sample corpus, then per-byte entropy on the demo sentence, then patches. A deliberately weak stand-in for BLT\u2019s learned entropy model \u2014 the mechanism is identical.

In [ ]:
import re

text = ("Byte models read raw bytes. Token models read chunks. "
        "Entropy patching groups predictable bytes together and splits at surprises.")

def norm(s):
    # toy simplification: lowercase, keep only a-z and space (27 symbols)
    return re.sub(r'[^a-z ]', ' ', s.lower())

corpus = norm("Language models predict the next token. Byte models predict the next byte. " * 8 +
              "Tokenizers split text into chunks. Entropy measures surprise. " * 8 +
              "When the next byte is predictable, bytes merge into a patch. " * 8 +
              "When entropy spikes at a boundary, a new patch begins. " * 8)
data  = np.frombuffer(norm(text).encode(), dtype=np.uint8)
cdata = np.frombuffer(corpus.encode(), dtype=np.uint8)

# byte-bigram with add-1 smoothing over the 27-symbol alphabet
idx = lambda b: 26 if b == 32 else b - 97   # space -> 26, 'a'..'z' -> 0..25
counts = np.ones((27, 27))
for a, b in zip(cdata[:-1], cdata[1:]):
    counts[idx(a), idx(b)] += 1
P = counts / counts.sum(axis=1, keepdims=True)

# per-position next-byte entropy H(b_{i+1} | b_i)
H = -(P[[idx(b) for b in data[:-1]]] * np.log2(P[[idx(b) for b in data[:-1]]])).sum(axis=1)
H = np.concatenate([[H[0]], H])

thr = np.median(H) + 0.75 * H.std()
bounds = [0] + [i for i in range(1, len(data)) if H[i] > thr] + [len(data)]
patches = [norm(text)[b0:b1] for b0, b1 in zip(bounds[:-1], bounds[1:])]

print(f'{len(data)} bytes -> {len(patches)} patches (threshold {thr:.2f} bits)')
print(' | '.join(patches))

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(range(len(data)), H, color='#4e79a7', width=1.0)
ax.axhline(thr, color='#e15759', ls='--', label=f'patch boundary threshold ({thr:.2f} bits)')
for b in bounds[1:-1]:
    ax.axvline(b - 0.5, color='#e15759', alpha=0.5, lw=0.8)
ax.set_xlabel('byte position'); ax.set_ylabel('next-byte entropy (bits)')
ax.set_title('Entropy spikes mark patch boundaries (BLT-style, toy 27-symbol bigram)')
ax.legend(fontsize=9); fig.tight_layout(); fig.savefig('/tmp/entropy_patches.png', dpi=120)
print('saved /tmp/entropy_patches.png')
print('\nReal BLT uses a small learned model for entropy and patches sub-word units;')
print('this toy bigram recovers word-ish patches, which is the same mechanism.')


## 3. Marginalize-It vs End-of-Token: the toy arithmetic

Setup: a teacher with a 4-token vocabulary. We have already observed bytes `t o k`. Two token paths are consistent with that prefix: `["token"]` (token *token* begins with `tok`) and `["tok"]` (token *tok*, then a boundary). The true byte-level distribution must respect which paths are possible — Marginalize-It ignores that structure; End-of-Token's explicit boundary marker preserves it exactly.

In [ ]:
# Teacher's next-token distribution given the context (toy numbers)
P_tok = {'token': 0.60, 'tok': 0.25, 'en': 0.10, 's': 0.05}
assert abs(sum(P_tok.values()) - 1.0) < 1e-9

# Paths consistent with observed bytes 'tok':
#   A: token 'token' (prob 0.60) -> next byte is 'e' with prob 1
#   B: token 'tok'   (prob 0.25) -> boundary; next byte drawn from fresh token distribution
pA = P_tok['token'] / (P_tok['token'] + P_tok['tok'])
pB = P_tok['tok'] / (P_tok['token'] + P_tok['tok'])

# P(next byte == 'e') after a boundary = sum of token probs for tokens starting with 'e'
p_e_after_boundary = P_tok['en']  # only 'en' starts with 'e' in this toy vocab

# TRUE byte-level conditional, respecting path structure
true_p_e = pA * 1.0 + pB * p_e_after_boundary

# MARGINALIZE-IT (approximate): sums token probs starting with 'e' over ALL tokens,
# ignoring which segmentations the tokenizer would actually emit at this position
marg_p_e = P_tok['token'] + P_tok['en']

# END-OF-TOKEN (exact): boundary probability is explicit via <eot>, so the
# byte conditional is transported losslessly -> equals the true value
eot_p_e = true_p_e

def kl(p, q):
    return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))

print(f'P(path A | "tok") = {pA:.4f},  P(path B | "tok") = {pB:.4f}')
print(f'True        P(next byte=e | "tok") = {true_p_e:.4f}')
print(f'Marginalize-It estimate               = {marg_p_e:.4f}  (KL vs true: {kl(true_p_e, marg_p_e):.4f} bits)')
print(f'End-of-Token estimate                 = {eot_p_e:.4f}  (KL vs true: {kl(true_p_e, eot_p_e):.4f} bits)')
print('\nMarginalize-It is lossy because it counts token paths the tokenizer')
print('would never emit here; EoT makes boundaries explicit, so nothing is lost.')

## 4. The crossover (schematic)

Qualitative reproduction of the paper's Figure 1 finding: token-distilled students lead at low compute and plateau; byte-distilled students cross over and keep climbing. **The +4% asymptote gap is the paper's fitted number; curve shapes and crossover placement are illustrative, not the paper's data.**

In [ ]:
flops = np.logspace(np.log10(3e19), np.log10(3e22), 400)
x = np.log10(flops)
pl = lambda x, a, s, k: a - s * np.exp(-(x - k) * 1.15)

tok     = pl(x, 52.6, 5.70, 19.40)
marg    = pl(x, 54.4, 7.90, 19.70)
eot     = pl(x, 54.7, 7.00, 19.95)
scratch = pl(x, 53.2, 6.25, 20.15)

# sanity: token leads early, byte crosses, asymptote gap ~+4% relative
i_lo, i_hi = 0, -1
assert tok[i_lo] > eot[i_lo], 'token model should lead at low compute'
assert eot[i_hi] > tok[i_hi], 'byte model should lead at high compute'
gap = (eot[i_hi] - tok[i_hi]) / tok[i_hi]
print(f'low-compute: token {tok[i_lo]:.1f} > byte {eot[i_lo]:.1f}  OK')
print(f'asymptote:  byte {eot[i_hi]:.1f} vs token {tok[i_hi]:.1f} -> +{gap*100:.1f}%  (paper: +4%) OK')
print(f'byte-from-scratch trails distilled byte: {scratch[i_hi]:.1f} < {eot[i_hi]:.1f}  OK')

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for y, lbl, c, ls in [(tok, 'Token-distilled (1B)', '#e15759', '-'),
                       (marg, 'Marginalize-It byte-distilled', '#59a14f', '--'),
                       (eot, 'End-of-Token byte-distilled', '#4e79a7', '-'),
                       (scratch, 'Byte model, from scratch', '#9c9c9c', ':')]:
    ax.plot(flops, y, label=lbl, color=c, ls=ls, lw=2.2)
ax.set_xscale('log'); ax.set_xlabel('Training compute (FLOPs)')
ax.set_ylabel('Avg downstream accuracy (%)')
ax.set_title('Schematic: tokens win early, bytes win the long game')
ax.legend(fontsize=9, loc='lower right')
fig.tight_layout(); fig.savefig('/tmp/scaling_crossover_nb.png', dpi=120)
print('saved /tmp/scaling_crossover_nb.png')

## Takeaways

- Tokenization is a **performance ceiling and a pricing decision**: it plateaus models early and taxes non-English scripts 2–4x per meaning.
- **Distillation is the bridge**: End-of-Token conversion moves a tokenized teacher's knowledge into a byte student losslessly, on ~1/6th the data.
- Go byte when the model is small, the training budget is long, and the workload is multilingual. Keep the tokenizer for short-budget English-only training.
- The paper's biggest gaps are **extrapolated scaling-law forecasts** — competitive today, dominant at the asymptote. Track the crossover, not the hype.